# Read the JSON and print the results.

In [1]:
import json
import os
import pandas as pd

# Define the path to your leaderboard JSON
file_path = "master_leaderboard.json"

def display_leaderboard_as_table(path):
    if not os.path.exists(path):
        return f"Error: File not found at {path}"

    with open(path, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            return "Error: Failed to decode JSON."

   
    # This makes 'epochs' its own column
    flattened_data = []
    for entry in data:
        row = entry.copy()
        params = row.pop('params', {})  
        row['epochs'] = params.get('epochs', 'N/A')
        flattened_data.append(row)


    df = pd.DataFrame(flattened_data)

   
    # Selecting and ordering the columns you want
    cols = ['dataset', 'features', 'model', 'epochs', 'test_f1', 'test_acc']
    df = df[cols]
    
    # Sort: Dataset A-Z, then Test F1 High to Low
    df = df.sort_values(by=['dataset', 'test_f1'], ascending=[True, False])

   
    df['test_f1'] = df['test_f1'].map('{:.4f}'.format)
    df['test_acc'] = df['test_acc'].map('{:.4f}'.format)

    
    # This will print like the Rating | Count table you showed
    print(df.to_string(index=False))

# Execute
if __name__ == "__main__":
    display_leaderboard_as_table(file_path)


       dataset                              features     model  epochs test_f1 test_acc
Amazon Ratings                              FastText GraphSAGE     100  0.4898   0.5440
Amazon Ratings                  amazon-mpnet-base-v2 GraphSAGE     200  0.4896   0.5475
Amazon Ratings                                 SBERT GraphSAGE     100  0.4638   0.5364
Amazon Ratings                  amazon-mpnet-base-v2     H2GCN     100  0.4271   0.4783
Amazon Ratings                              FastText     H2GCN     100  0.4212   0.4977
Amazon Ratings                                 SBERT     H2GCN     100  0.4190   0.4954
Amazon Ratings                  amazon-mpnet-base-v2       GAT     200  0.4046   0.4834
Amazon Ratings                                 SBERT       GCN     100  0.3982   0.4756
Amazon Ratings                  amazon-mpnet-base-v2       GCN     100  0.3891   0.4732
Amazon Ratings                                 SBERT      GCN2     400  0.3683   0.4613
Amazon Ratings                  

In [2]:
import json
import os
import pandas as pd

file_path = "master_leaderboard.json"

def display_best_combinations(path):
    if not os.path.exists(path):
        return f"Error: File not found at {path}"

    with open(path, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            return "Error: Failed to decode JSON."

   
    flattened_data = []
    for entry in data:
        row = entry.copy()
        params = row.pop('params', {})
        row['epochs'] = params.get('epochs', 'N/A')
        flattened_data.append(row)

    df = pd.DataFrame(flattened_data)

    
    # We group by those three columns and find the index of the maximum 'test_f1'
    idx = df.groupby(['dataset', 'features', 'model'])['test_f1'].idxmax()
    best_df = df.loc[idx].copy()

    
    best_df = best_df.sort_values(by=['dataset', 'features'], ascending=[True, True])

   
    cols = ['dataset', 'features', 'model', 'epochs', 'test_f1', 'test_acc']
    best_df = best_df[cols]
    
    best_df['test_f1'] = best_df['test_f1'].map('{:.4f}'.format)
    best_df['test_acc'] = best_df['test_acc'].map('{:.4f}'.format)

   
   
    print(best_df.to_string(index=False))

if __name__ == "__main__":
    display_best_combinations(file_path)


       dataset                              features     model  epochs test_f1 test_acc
Amazon Ratings                              FastText       GAT     100  0.3497   0.4572
Amazon Ratings                              FastText       GCN     100  0.3584   0.4672
Amazon Ratings                              FastText      GCN2     400  0.3480   0.4644
Amazon Ratings                              FastText GraphSAGE     100  0.4898   0.5440
Amazon Ratings                              FastText     H2GCN     100  0.4212   0.4977
Amazon Ratings                                 SBERT       GAT     100  0.3675   0.4730
Amazon Ratings                                 SBERT       GCN     100  0.3982   0.4756
Amazon Ratings                                 SBERT      GCN2     400  0.3683   0.4613
Amazon Ratings                                 SBERT GraphSAGE     100  0.4638   0.5364
Amazon Ratings                                 SBERT     H2GCN     100  0.4190   0.4954
Amazon Ratings                  

In [4]:
import json
import os
import pandas as pd

file_path = "master_leaderboard.json"

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def create_comparison_tables(path):
    if not os.path.exists(path):
        return f"Error: File not found at {path}"

    with open(path, 'r') as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    # Clean up model names
    df['model'] = df['model'].astype(str).str.replace(
        r'.*Custom BERT.*Roman Empire.*', 'Custom BERT', regex=True, case=False
    )

    def get_dataset_view(dataset_name):
        subset = df[df['dataset'] == dataset_name].copy()
        if subset.empty: return None

        # Pivot to get max values
        pivot_f1 = subset.pivot_table(index='model', columns='features', values='test_f1', aggfunc='max')
        pivot_acc = subset.pivot_table(index='model', columns='features', values='test_acc', aggfunc='max')

        view = pd.DataFrame(index=pivot_f1.index, columns=pivot_f1.columns, dtype=object)

        for col in pivot_f1.columns:
            for idx in pivot_f1.index:
                f1, acc = pivot_f1.loc[idx, col], pivot_acc.loc[idx, col]
                if pd.isna(f1):
                    view.loc[idx, col] = "      -      "
                else:
                    view.loc[idx, col] = f"{f1*100:.2f} / {acc*100:.2f}"
        
        view.index.name = "MODEL"
        return view

    # --- THE KEY CHANGE ---
    # Automatically get every unique dataset name present in your JSON
    unique_datasets = sorted(df['dataset'].unique())

    for ds in unique_datasets:
        print("\n" + "="*80)
        print(f"  {ds.upper()} LEADERBOARD (Best F1 / Acc)")
        print("="*80)
        view = get_dataset_view(ds)
        if view is not None:
            print(view.sort_index(axis=1).to_string())

if __name__ == "__main__":
    create_comparison_tables(file_path)



  AMAZON RATINGS LEADERBOARD (Best F1 / Acc)
features        FastText          SBERT amazon-mpnet-base-v2
MODEL                                                       
GAT        34.97 / 45.72  36.75 / 47.30        40.46 / 48.34
GCN        35.84 / 46.72  39.82 / 47.56        38.91 / 47.32
GCN2       34.80 / 46.44  36.83 / 46.13        36.67 / 46.76
GraphSAGE  48.98 / 54.40  46.38 / 53.64        48.96 / 54.75
H2GCN      42.12 / 49.77  41.90 / 49.54        42.71 / 47.83

  CORA LEADERBOARD (Best F1 / Acc)
features             PyG
MODEL                   
GAT        84.78 / 85.70
GCN        86.76 / 87.80
GCN2       82.99 / 83.90
GraphSAGE  85.08 / 86.20
H2GCN      82.68 / 84.00

  CORNELL LEADERBOARD (Best F1 / Acc)
features             PyG
MODEL                   
GAT        28.49 / 43.24
GCN        32.97 / 45.95
GCN2       37.22 / 64.86
GraphSAGE  63.17 / 75.68
H2GCN      60.49 / 78.38

  PUBMED LEADERBOARD (Best F1 / Acc)
features             PyG
MODEL                   
GAT        84.